# Digital experience monitoring: When synthetic, RUM and APM disagree

Companion notebook for [the article](https://www.elastic.co/observability-labs/blog/digital-experience-monitoring-synthetic-rum-slo).

It reads the data from the companion lab in this folder, so run the lab first (`README.md`) and set `LAB_T0` in `.env` to the `T0` you passed to `scenario.py`. The API key needs to read `traces-*`, `logs-*` and `synthetics-*`, and to create SLOs and rules. To create one in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "digital-experience-monitoring-synthetic-rum-slo-notebook", "role_descriptors": {} }
```

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import os
from datetime import datetime, timedelta

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"
NS = os.getenv("LAB_NAMESPACE", "dem")
T0 = datetime.fromisoformat(os.environ["LAB_T0"].replace("Z", "+00:00"))
MONITOR_ID = os.getenv(
    "SYNTHETICS_MONITOR_ID", "dem-checkout-journey-bean-box-dem-default"
)
SCALE = float(
    os.getenv("LAB_TIME_SCALE", "1")
)  # the TIME_SCALE you gave scenario.py, if any

TRACES = f"traces-generic.otel-{NS}"
LOGS = f"logs-generic.otel-{NS}"
NOT_SYNTHETIC = 'NOT user_agent.original LIKE "*Elastic/Synthetics*"'
TAG = "dem-notebook"  # every SLO and rule this notebook creates carries this tag; Clean up uses it

# Fault windows, in minutes after T0, from scenario.py (on, off)
FAULTS = {
    "Live gateway hangs": (45, 75),
    "3 MB image on phones": (105, 145),
    "Sandbox down": (175, 205),
    "Slow orders": (235, 275),
    "Safari-only bug": (305, 345),
    "Payment provider down": (375, 405),
}

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
    "elastic-api-version": "2023-10-31",
}


def kbn(method, path, **kwargs):
    """Call a Kibana API in the configured space and raise on HTTP errors."""
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=120, **kwargs
    )
    if not response.ok:
        print(response.text)
    response.raise_for_status()
    return response.json() if response.text else {}


def window(fault):
    """Start and end of a fault window as ISO strings."""
    start, end = FAULTS[fault]
    return (
        (T0 + timedelta(minutes=start * SCALE)).isoformat(),
        (T0 + timedelta(minutes=end * SCALE)).isoformat(),
    )


def time_filter(fault=None):
    start, end = (
        window(fault)
        if fault
        else (T0.isoformat(), (T0 + timedelta(minutes=415 * SCALE)).isoformat())
    )
    return {"range": {"@timestamp": {"gte": start, "lt": end}}}


def esql(query, fault=None, title=None):
    """Run an ES|QL query over a fault window (or the whole run) and print it as a table."""
    response = es.esql.query(query=query, filter=time_filter(fault), format="txt")
    if title:
        print(f"\n{title}")
    print(response.body)


def esql_value(query, fault):
    """Run an ES|QL query that returns one row with one column, and return that value."""
    response = es.esql.query(query=query, filter=time_filter(fault))
    values = response.body["values"]
    return values[0][0] if values else None


print(
    f"elasticsearch {es.info()['version']['number']} | space={SPACE} | namespace={NS} | T0={T0.isoformat()}"
)
for fault in FAULTS:
    print(f"{fault:24} {window(fault)[0]} -> {window(fault)[1]}")

## 2. Put an SLO on each signal

SLO history only covers the last 7 days. The ES|QL in the next sections works on any past run.

In [ ]:
BURN_RATE_WINDOWS = [  # Kibana's defaults for a 7-day SLO
    {
        "id": "page-fast",
        "actionGroup": "slo.burnRate.alert",
        "burnRateThreshold": 3.36,
        "maxBurnRateThreshold": 168,
        "longWindow": {"value": 1, "unit": "h"},
        "shortWindow": {"value": 5, "unit": "m"},
    },
    {
        "id": "page-slow",
        "actionGroup": "slo.burnRate.high",
        "burnRateThreshold": 1.4,
        "maxBurnRateThreshold": 28,
        "longWindow": {"value": 6, "unit": "h"},
        "shortWindow": {"value": 30, "unit": "m"},
    },
    {
        "id": "ticket-fast",
        "actionGroup": "slo.burnRate.medium",
        "burnRateThreshold": 0.7,
        "maxBurnRateThreshold": 7,
        "longWindow": {"value": 24, "unit": "h"},
        "shortWindow": {"value": 120, "unit": "m"},
    },
    {
        "id": "ticket-slow",
        "actionGroup": "slo.burnRate.low",
        "burnRateThreshold": 0.234,
        "maxBurnRateThreshold": 2.3,
        "longWindow": {"value": 72, "unit": "h"},
        "shortWindow": {"value": 360, "unit": "m"},
    },
]


def create_slo(body):
    body = {
        "timeWindow": {"duration": "7d", "type": "rolling"},
        "budgetingMethod": "occurrences",
        "objective": {"target": 0.99},
        **body,
        "tags": ["bean-box", TAG],
    }
    slo_id = kbn("POST", "/api/observability/slos", json=body)["id"]
    rule = {
        "name": f"{body['name']} burn rate",
        "rule_type_id": "slo.rules.burnRate",
        "consumer": "slo",
        "schedule": {"interval": "1m"},
        "tags": ["bean-box", TAG],
        "params": {"sloId": slo_id, "windows": BURN_RATE_WINDOWS},
        "actions": [],
    }
    kbn("POST", "/api/alerting/rule", json=rule)
    print(f"created SLO {body['name']!r} ({slo_id}) and its burn rate rule")
    return slo_id


BROWSER_FILTER = 'service.name : "bean-box-web" and url.full : */api/orders'

SLOS = {}
SLOS["journey"] = create_slo(
    {
        "name": "Checkout journey availability",
        "description": "Share of synthetic checkout journeys that complete an order with the test card.",
        "indicator": {
            "type": "sli.synthetics.availability",
            "params": {
                "index": "synthetics-*",
                "monitorIds": [{"value": MONITOR_ID, "label": "Checkout journey"}],
                "projects": [],
                "tags": [],
                "filter": "",
            },
        },
    }
)
SLOS["apm"] = create_slo(
    {
        "name": "Order API latency",
        "description": "Share of POST /api/orders requests that the shop answers in under 1 second.",
        "indicator": {
            "type": "sli.apm.transactionDuration",
            "params": {
                "service": "bean-box-shop",
                "environment": "dem-lab",
                "transactionType": "request",
                "transactionName": "POST /api/orders",
                "threshold": 1000,
                "filter": "",
                "index": "metrics-apm*,apm-*,metrics-*.otel-*",
            },
        },
    }
)
SLOS["browser"] = create_slo(
    {
        "name": "Checkout success in the browser",
        "description": "Share of order requests sent from browsers (EDOT Browser fetch spans) that succeed.",
        "indicator": {
            "type": "sli.kql.custom",
            "params": {
                "index": TRACES,
                "filter": BROWSER_FILTER,
                "good": 'event.outcome : "success"',
                "total": "",
                "timestampField": "@timestamp",
            },
        },
    }
)

## 3. Split synthetic and real orders in one query

In [ ]:
SPLIT_ORDERS = f"""
FROM {TRACES}
| WHERE service.name == "bean-box-web" AND url.full LIKE "*/api/orders"
| EVAL source = CASE(user_agent.original LIKE "*Elastic/Synthetics*", "synthetic journey", "visitors")
| STATS orders = COUNT(*),
        failed = COUNT(*) WHERE event.outcome == "failure",
        p95_ms = ROUND(PERCENTILE(duration, 95) / 1000000)
    BY source
| LIMIT 10
"""
esql(SPLIT_ORDERS, title="Whole run")

## 4. Measure every signal in a fault window

In [ ]:
SIGNALS = {
    "Synthetic: checks passed": (
        0.99,
        f"""
        FROM synthetics-* | WHERE monitor.id == "{MONITOR_ID}" AND synthetics.type == "heartbeat/summary"
        | STATS ok = COUNT(*) WHERE monitor.status == "up", n = COUNT(*) | EVAL v = 100.0 * ok / n | KEEP v | LIMIT 1""",
    ),
    "RUM: visitors' orders succeeded": (
        0.99,
        f"""
        FROM {TRACES} | WHERE service.name == "bean-box-web" AND url.full LIKE "*/api/orders" AND {NOT_SYNTHETIC}
        | STATS ok = COUNT(*) WHERE event.outcome == "success", n = COUNT(*) | EVAL v = 100.0 * ok / n | KEEP v | LIMIT 1""",
    ),
    "RUM: all browser orders succeeded": (
        0.99,
        f"""
        FROM {TRACES} | WHERE service.name == "bean-box-web" AND url.full LIKE "*/api/orders"
        | STATS ok = COUNT(*) WHERE event.outcome == "success", n = COUNT(*) | EVAL v = 100.0 * ok / n | KEEP v | LIMIT 1""",
    ),
    "APM: orders under 1 s": (
        0.99,
        f"""
        FROM {TRACES} | WHERE service.name == "bean-box-shop" AND name == "POST /api/orders"
        | STATS ok = COUNT(*) WHERE duration < 1000000000, n = COUNT(*) | EVAL v = 100.0 * ok / n | KEEP v | LIMIT 1""",
    ),
    "RUM: phone LCP under 2.5 s": (
        0.95,
        f"""
        FROM {LOGS} | WHERE event_name == "browser.web_vital" AND attributes.browser.web_vital.name == "lcp"
          AND resource.attributes.browser.mobile == true AND {NOT_SYNTHETIC}
        | STATS ok = COUNT(*) WHERE attributes.browser.web_vital.value <= 2500, n = COUNT(*)
        | EVAL v = 100.0 * ok / n | KEEP v | LIMIT 1""",
    ),
}
ERRORS = f"""FROM {LOGS} | WHERE event_name == "exception" AND service.name == "bean-box-web" | STATS v = COUNT(*) | LIMIT 1"""


def signal_check(fault, show=True):
    """Good share and burn rate per signal for one fault window."""
    out = print if show else (lambda *a, **k: None)
    out(f"{fault}  ({window(fault)[0]} -> {window(fault)[1]})")
    out(f"{'signal':36} {'good %':>8} {'burn rate':>10}  state")
    states = {}
    for name, (target, query) in SIGNALS.items():
        good = esql_value(query, fault)
        if good is None:
            out(f"{name:36} {'no data':>8}")
            continue
        burn = (100 - good) / (100 * (1 - target))
        state = "RED" if good < 100 * target else "green"
        states[name] = state
        out(f"{name:36} {good:8.1f} {burn:10.1f}  {state}")
    errors = esql_value(ERRORS, fault) or 0
    states["RUM: browser errors"] = "RED" if errors > 0 else "green"
    out(f"{'RUM: browser errors (count)':36} {errors:8}")
    return states

## 5. The live payment gateway hangs

In [ ]:
esql(SPLIT_ORDERS, "Live gateway hangs", title="Orders by source")
signal_check("Live gateway hangs");

## 6. A 3 MB image on phones

In [ ]:
esql(
    f"""
FROM {LOGS}
| WHERE event_name == "browser.web_vital" AND attributes.browser.web_vital.name == "lcp"
| EVAL source = CASE(user_agent.original LIKE "*Elastic/Synthetics*", "synthetic journey", "visitors")
| STATS lcp_p75 = ROUND(PERCENTILE(attributes.browser.web_vital.value, 75)), records = COUNT(*)
    BY source, mobile = resource.attributes.browser.mobile, browser = resource.attributes.browser.name
| SORT lcp_p75 DESC
| LIMIT 10
""",
    "3 MB image on phones",
    title="LCP by device and browser",
)
signal_check("3 MB image on phones")

SLOS["lcp"] = create_slo(
    {
        "name": "Page load speed (LCP) by device",
        "description": "Share of visitor page loads with LCP at or under 2.5 s, per device class.",
        "indicator": {
            "type": "sli.kql.custom",
            "params": {
                "index": LOGS,
                "filter": 'service.name : "bean-box-web" and event_name : "browser.web_vital" and '
                'attributes.browser.web_vital.name : "lcp" and not user_agent.original : *Elastic/Synthetics*',
                "good": "attributes.browser.web_vital.value <= 2500",
                "total": "",
                "timestampField": "@timestamp",
            },
        },
        "groupBy": "resource.attributes.browser.mobile",
        "objective": {"target": 0.95},
    }
)

## 7. The payment sandbox is down

In [ ]:
signal_check("Sandbox down")

slo = kbn("GET", f"/api/observability/slos/{SLOS['browser']}")
indicator = slo["indicator"]
indicator["params"]["filter"] = (
    BROWSER_FILTER + " and not user_agent.original : *Elastic/Synthetics*"
)
kbn("PUT", f"/api/observability/slos/{SLOS['browser']}", json={"indicator": indicator})
print(
    "browser SLO filter:",
    kbn("GET", f"/api/observability/slos/{SLOS['browser']}")["indicator"]["params"][
        "filter"
    ],
)

## 8. Slow orders after the click

In [ ]:
esql(
    f"""
FROM {TRACES}
| WHERE service.name == "bean-box-shop" AND name == "POST /api/orders"
| STATS requests = COUNT(*),
        over_1s = COUNT(*) WHERE duration >= 1000000000,
        p95_ms = ROUND(PERCENTILE(duration, 95) / 1000000)
| LIMIT 10
""",
    "Slow orders",
    title="POST /api/orders in APM",
)
signal_check("Slow orders");

## 9. A Safari-only checkout bug

In [ ]:
esql(
    f"""
FROM {LOGS}
| WHERE event_name == "exception" AND service.name == "bean-box-web"
| STATS errors = COUNT(*), sessions = COUNT_DISTINCT(attributes.session.id)
    BY browser = resource.attributes.browser.name, message = attributes.exception.message
| SORT errors DESC
| LIMIT 10
""",
    "Safari-only bug",
    title="Browser errors by browser",
)
signal_check("Safari-only bug")

ERRORS_RULE = kbn(
    "POST",
    "/api/alerting/rule",
    json={
        "name": "Browser errors by browser",
        "rule_type_id": "observability.rules.custom_threshold",
        "consumer": "logs",
        "schedule": {"interval": "1m"},
        "tags": ["bean-box", TAG],
        "params": {
            "criteria": [
                {
                    "comparator": ">",
                    "metrics": [{"name": "A", "aggType": "count"}],
                    "threshold": [5],
                    "timeSize": 10,
                    "timeUnit": "m",
                }
            ],
            "alertOnNoData": False,
            "alertOnGroupDisappear": False,
            "groupBy": ["resource.attributes.browser.name"],
            "searchConfiguration": {
                "index": {"title": LOGS, "timeFieldName": "@timestamp"},
                "query": {
                    "query": 'service.name : "bean-box-web" and event_name : "exception"',
                    "language": "kuery",
                },
            },
        },
        "actions": [],
    },
)["id"]
print(f"created rule 'Browser errors by browser' ({ERRORS_RULE})")

## 10. The payment provider is down

In [ ]:
esql(SPLIT_ORDERS, "Payment provider down", title="Orders by source")
signal_check("Payment provider down");

## 11. Which signal should you believe?

In [ ]:
rows = {fault: signal_check(fault, show=False) for fault in FAULTS}
columns = [
    "Synthetic: checks passed",
    "RUM: visitors' orders succeeded",
    "APM: orders under 1 s",
    "RUM: phone LCP under 2.5 s",
    "RUM: browser errors",
]
short = ["Synthetic", "RUM orders", "APM", "RUM phone LCP", "RUM errors"]
print(f"{'Failure':24}" + "".join(f"{c:>15}" for c in short))
for fault, states in rows.items():
    print(f"{fault:24}" + "".join(f"{states.get(c, '-'):>15}" for c in columns))

## 12. Clean up

Deletes the SLOs and rules tagged `dem-notebook`. The lab `README.md` covers the rest.

In [ ]:
rules = kbn(
    "GET",
    "/api/alerting/rules/_find",
    params={"filter": f'alert.attributes.tags:"{TAG}"', "per_page": 100},
)
for rule in rules["data"]:
    kbn("DELETE", f"/api/alerting/rule/{rule['id']}")
    print(f"deleted rule {rule['name']!r}")

slos = kbn(
    "GET",
    "/api/observability/slos",
    params={"kqlQuery": f'slo.tags:"{TAG}"', "perPage": 100},
)
for slo in slos["results"]:
    kbn("DELETE", f"/api/observability/slos/{slo['id']}")
    print(f"deleted SLO {slo['name']!r}")
print("done")